In [ ]:
!nvidia-smi

Fri May 29 17:30:09 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install nvcc4jupyter

In [ ]:
%load_ext nvcc4jupyter

Detected platform "Colab". Running its setup...
Source files will be saved in "/tmp/tmp4shvry2k".


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving goog_vol_surface.csv to goog_vol_surface.csv


In [ ]:
!unzip nvidia-mathd*.zip -d /usr/local/nvidia-math

Streaming output truncated to the last 5000 lines.
  inflating: /usr/local/nvidia-math/nvidia-mathdx-25.12.1-cuda12/nvidia/mathdx/25.12/example/cufftdx/03_block_fft_performance/block_fft_performance_many.cu  
  inflating: /usr/local/nvidia-math/__MACOSX/nvidia-mathdx-25.12.1-cuda12/nvidia/mathdx/25.12/example/cufftdx/03_block_fft_performance/._block_fft_performance_many.cu  
  inflating: /usr/local/nvidia-math/nvidia-mathdx-25.12.1-cuda12/nvidia/mathdx/25.12/example/cufftdx/03_block_fft_performance/block_fft_performance.cu  
  inflating: /usr/local/nvidia-math/__MACOSX/nvidia-mathdx-25.12.1-cuda12/nvidia/mathdx/25.12/example/cufftdx/03_block_fft_performance/._block_fft_performance.cu  
  inflating: /usr/local/nvidia-math/nvidia-mathdx-25.12.1-cuda12/nvidia/mathdx/25.12/include/commondx/operators/type.hpp  
  inflating: /usr/local/nvidia-math/__MACOSX/nvidia-mathdx-25.12.1-cuda12/nvidia/mathdx/25.12/include/commondx/operators/._type.hpp  
  inflating: /usr/local/nvidia-math/nvidia-mathd

In [ ]:
!cp -r /usr/local/nvidia-math/nvidia-mathdx-25.12.1-cuda12/nvidia/mathdx/25.12/include/* /usr/local/include/

In [ ]:
%%cuda -c "-O2 -arch=sm_75"
// NEW VERSION
#include <curanddx.hpp>
#include <iostream>
#include <fstream>
#include <vector>
#include <string>
#include <sstream>
#include <iomanip>
#include <algorithm>
#include <random>
#include <iostream>
#include <iomanip> // Necessary for setw()
#include <cmath>
#include <ctime>
#include <chrono>

struct OptionGrid {
    std::vector<float> strikes;
    std::vector<std::string> expiries;
    std::vector<std::vector<float>> prices;
    size_t num_rows = 0;
    size_t num_cols = 0;
    std::vector<std::vector<float>> option_prices;
};

OptionGrid load_csv(const std::string& filename) {
    OptionGrid option_grid;
    std::ifstream file(filename);

    if (!file.is_open()) {
        std::cerr << "Error: Could not open file " << filename << std::endl;
        return option_grid;
    }

    std::string line;

    // 1. Parse Header Row (Strikes)
    if (std::getline(file, line)) {

        int i = line.find(',') + 1;

        while (i < line.length()) {
            if (line[i] != ',' && line[i] != '\r' && line[i] != '\n') {
                std::string s = "";
                while (i < line.length() && line[i] != ',') {
                    s += line[i];
                    i++;
                }
                if (!s.empty()) option_grid.strikes.push_back(std::stof(s));
            }
            i++;
        }
    }

    int price_row = 0;
    while (std::getline(file, line)) {
        if (line.empty()) continue;

        int pos = line.find(',');
        option_grid.expiries.push_back(line.substr(0, pos));

        int i = pos + 1;
        option_grid.prices.emplace_back();

        while (i < line.length()) {
            if (line[i] == ',') {
                option_grid.prices[price_row].push_back(0.0); // Treat as 0 or NaN
                i++;
                continue;
            }

            std::string s = "";
            while (i < line.length() && line[i] != ',') {
                s += line[i];
                i++;
            }

            try {
                if (s == "NaN" || s == "") option_gridprices[price_row].push_back(0.0);
                else option_grid.prices[price_row].push_back(std::stof(s));
            } catch (...) {
                option_grid.prices[price_row].push_back(0.0);
            }
            i++;
        }
        price_row++;
    }

    option_grid.num_rows = option_grid.expiries.size();
    option_grid.num_cols = option_grid.strikes.size();
    return option_grid;
}
long days_between(std::string s1, std::string s2) {
        struct std::tm tm1 = {0}, tm2 = {0};
        std::istringstream ss1(s1), ss2(s2);

        // Parse the strings
        ss1 >> std::get_time(&tm1, "%Y-%m-%d");
        ss2 >> std::get_time(&tm2, "%Y-%m-%d");

        // Convert to time_t (seconds since 1970)
        std::time_t time1 = std::mktime(&tm1);
        std::time_t time2 = std::mktime(&tm2);

        // Calculate difference in seconds and convert to days
        const long seconds_per_day = 60 * 60 * 24;
        return (time2 - time1) / seconds_per_day;
}

//Annual vol, annual int_rate, stock price
__constant__ double annual_interest_rate = 0.07;
__constant__ double annual_vol = 0.2;
__constant__ double stock_price = 382.09;

__constant__ double2 option_arr[918];

constexpr int num_threads_per_block = 256; //1
constexpr int num_blocks_per_option = 512; //1
constexpr int num_threads_per_option = num_threads_per_block * num_blocks_per_option; // This will give over 100,000 samples per option
constexpr int num_options = 918; // 18 expirations * 51 strike prices = 918 options
constexpr int total_threads = num_options * num_threads_per_option;

//This is syntax for the type of random number generation
// philox is a rigorous, non-cryptographic type of number generation
// 800 is the device architecture (Turing T4)

using RNG = decltype(curanddx::Generator<curanddx::philox4_32>() +
                     curanddx::SM<750>() +
                     curanddx::Thread());


__global__ void monte_carlo_gpu_p1(double* options_payoffs,
                                   double* options_payoffs_2,
                                   const unsigned long long seed,
                                   const typename RNG::offset_type rng_offset)
{
    int i = blockDim.x * blockIdx.x + threadIdx.x;
    if (i > total_threads) return;
    int option_id = i / num_threads_per_option;

    //gen random number

    RNG rng(seed, ((rng_offset + i) % 65536), ((rng_offset + i) / 65536));
    curanddx::normal<double, curanddx::box_muller> dist(0.0, 1.0);
    double4 samples = dist.generate4(rng);
    double sample = samples.x;

    //need time to expiration calculation, and strike price

    double time_delta = option_arr[option_id].x;
    double strike_price = option_arr[option_id].y;


    // then i can calc vol, and use stock price

    double volatility = annual_vol * sqrt(time_delta);

    //then I can generate the price

    double random_price = stock_price * exp((annual_interest_rate - 0.5 * annual_vol * annual_vol) * time_delta + annual_vol * sqrt(time_delta)  * sample);

    // then the payoff
    double option_payoff = max(0.0,random_price - strike_price);
    options_payoffs[i] = option_payoff;

    //sync threads

    __syncthreads();

    // do par reduction
    // The following assumes 256 threads per block

    int offset = threadIdx.x % num_threads_per_block;

    if (offset < 128) { options_payoffs[i] += options_payoffs[i + 128]; __syncthreads(); }
    if (offset < 64) { options_payoffs[i] += options_payoffs[i + 64]; __syncthreads(); }
    if (offset < 32) { options_payoffs[i] += options_payoffs[i + 32]; __syncthreads(); }
    if (offset < 16) { options_payoffs[i] += options_payoffs[i + 16]; __syncthreads(); }
    if (offset < 8) { options_payoffs[i] += options_payoffs[i + 8]; __syncthreads(); }
    if (offset < 4) { options_payoffs[i] += options_payoffs[i + 4]; __syncthreads(); }
    if (offset < 2) { options_payoffs[i] += options_payoffs[i + 2]; __syncthreads(); }
    if (offset < 1) { int option_sum_index = blockIdx.x; //num_blocks_per_option * option_id + blockIdx.x % num_blocks_per_option;
                      options_payoffs_2[option_sum_index] = options_payoffs[i] + options_payoffs[i + 1];
                      __syncthreads();
                    }
    /*
    At this point, options_payoffs contains 512 unique sums of 256 unique potential payoffs for option 0 in index 0, 256, ..., 511*256
    option 2 contains the payoffs for option 1 in index 512* 256 and so on
    */


}

__global__ void monte_carlo_gpu_p2(double* options_payoffs_2,
                                   double* options_values) {
    int i = blockDim.x * blockIdx.x + threadIdx.x;
    //The following assumes 256 threads per block, 512 blocks per option
    int option_id = i / 512;
    int offset = i % 512;
    double time_delta = option_arr[option_id].x;
    double strike_price = option_arr[option_id].y;

    if (offset < 256) { options_payoffs_2[i] += options_payoffs_2[i + 256]; __syncthreads(); }
    if (offset < 128) { options_payoffs_2[i] += options_payoffs_2[i + 128]; __syncthreads(); }
    if (offset < 64)  { options_payoffs_2[i] += options_payoffs_2[i + 64];  __syncthreads(); }
    if (offset < 32)  { options_payoffs_2[i] += options_payoffs_2[i + 32];  __syncthreads(); }
    if (offset < 16)  { options_payoffs_2[i] += options_payoffs_2[i + 16];  __syncthreads(); }
    if (offset < 8)   { options_payoffs_2[i] += options_payoffs_2[i + 8];   __syncthreads(); }
    if (offset < 4)   { options_payoffs_2[i] += options_payoffs_2[i + 4];   __syncthreads(); }
    if (offset < 2)   { options_payoffs_2[i] += options_payoffs_2[i + 2];   __syncthreads(); }
    //After the following code, option values are placed in indices 0, 256, 512, ..., 917* 256 in options_values
    if (offset < 1) {
        double time_delta = option_arr[option_id].x;
        options_values[option_id] = exp(-annual_interest_rate * time_delta)*((options_payoffs_2[i] + options_payoffs_2[i + 1])/(512*256));
        //printf("i: %d, td: %f, strike: %f, price: %f\n", i, time_delta, strike_price, options_values[i]);
        __syncthreads();
    }

}
double* options_payoffs;
double* options_payoffs_2;
double* options_values;
int main() {
    std::string path = "goog_vol_surface.csv";
    std::cout << "Loading: " << path << "..." << std::endl;
    OptionGrid option_grid = load_csv(path);
    option_grid.option_prices.resize(option_grid.num_rows, std::vector<float>(option_grid.num_cols, 0.0));

    // Monte Carlo GPU Time
    auto start = std::chrono::high_resolution_clock::now();


    double2 h_option_arr[918];
    for (int i = 0; i < option_grid.num_rows; i++) {
        double time_delta = days_between("2026-05-13",option_grid.expiries[i]) / 365.0;
        for (int j = 0; j < option_grid.num_cols; j++) {
            double strike_price = option_grid.strikes[j];
            h_option_arr[i*option_grid.num_cols + j] = {time_delta,strike_price};
        }
    }

    constexpr int num_blocks = num_blocks_per_option * num_options;

    cudaMemcpyToSymbol(option_arr, h_option_arr, 918 * sizeof(double2));

    cudaMalloc(&options_payoffs, total_threads * sizeof(double));

    cudaMalloc(&options_payoffs_2, num_options * num_blocks_per_option * sizeof(double));

    cudaMalloc(&options_values, num_options * sizeof(double));

    unsigned long long seed = 1234ULL;
    typename RNG::offset_type rng_offset = 0;

    monte_carlo_gpu_p1<<<num_blocks,num_threads_per_block>>>(options_payoffs, options_payoffs_2, seed, rng_offset);
    cudaDeviceSynchronize();

    monte_carlo_gpu_p2<<<num_options,num_blocks_per_option>>>(options_payoffs_2, options_values);
    cudaDeviceSynchronize();

    double h_options_values[918];

    cudaMemcpy(
    h_options_values,
    options_values,
    918 * sizeof(double),
    cudaMemcpyDeviceToHost
    );

    for (int i = 0; i < 918; i ++) {
        double time_delta = h_option_arr[i].x;
        double strike_price = h_option_arr[i].y;
        std::cout << "Td: " << time_delta << ", k: " << strike_price << ", value: " << h_options_values[i] << '\n';
    }

    auto end = std::chrono::high_resolution_clock::now();
    std::chrono::duration<double> elapsed = end - start;
    std::cout << "Load time: " << elapsed.count() << " seconds." << std::endl;

    return 0;
}

Loading: goog_vol_surface.csv...
Td: 0.0246575, k: 170, value: 212.416
Td: 0.0246575, k: 175, value: 207.399
Td: 0.0246575, k: 180, value: 202.376
Td: 0.0246575, k: 185, value: 197.371
Td: 0.0246575, k: 195, value: 187.387
Td: 0.0246575, k: 200, value: 182.429
Td: 0.0246575, k: 205, value: 177.399
Td: 0.0246575, k: 210, value: 172.461
Td: 0.0246575, k: 220, value: 162.439
Td: 0.0246575, k: 225, value: 157.552
Td: 0.0246575, k: 230, value: 152.483
Td: 0.0246575, k: 240, value: 142.562
Td: 0.0246575, k: 245, value: 137.55
Td: 0.0246575, k: 250, value: 132.533
Td: 0.0246575, k: 255, value: 127.515
Td: 0.0246575, k: 260, value: 122.557
Td: 0.0246575, k: 270, value: 112.58
Td: 0.0246575, k: 275, value: 107.556
Td: 0.0246575, k: 280, value: 102.566
Td: 0.0246575, k: 285, value: 97.5451
Td: 0.0246575, k: 290, value: 92.5611
Td: 0.0246575, k: 295, value: 87.6564
Td: 0.0246575, k: 300, value: 82.6705
Td: 0.0246575, k: 305, value: 77.6306
Td: 0.0246575, k: 310, value: 72.6273
Td: 0.0246575, k: 3